# 04 · 索引与掩码 Indexing & Masking

> **能力标签**：SH3（NumPy 与向量化计算 / NumPy & vectorized computing）

## 目标 Objectives

完成本课后，你应该能够：

1. 使用基本切片（basic slicing）获取数组的**视图（view）**，理解视图与拷贝（copy）的区别。
2. 使用**花式索引（fancy indexing）**（整数数组）和**布尔索引（boolean indexing）**选取元素。
3. 用 `np.where` 实现条件选择（三元操作）。
4. 正确区分何时需要 `.copy()`，避免意外修改原数组。

> 对应能力：**SH3**


## 概念讲解 Concepts

### 基本切片 Basic Slicing（视图）

基本切片使用 `start:stop:step` 语法，返回**视图（view）**——与原数组共享内存：

```python
a = np.arange(10)
v = a[2:7:2]     # [2, 4, 6] — view
v[0] = 99        # 修改 v → a 也变！
```

要得到独立拷贝：`v = a[2:7:2].copy()`

---

### 花式索引 Fancy Indexing（总是拷贝）

用整数数组或列表索引，返回**拷贝**：

```python
a = np.arange(10)
idx = np.array([0, 3, 7])
b = a[idx]       # [0, 3, 7] — copy
b[0] = 99        # 不影响 a
```

二维花式索引：`A[[0, 2], :]`（选第 0、2 行）

---

### 布尔索引 Boolean Indexing（总是拷贝）

用布尔数组（mask）选取满足条件的元素：

```python
a = np.array([1, -2, 3, -4, 5])
mask = a > 0          # [True, False, True, False, True]
a[mask]               # [1, 3, 5]
```

常见模式：

```python
a[a > 0]              # 等价于上面
a[(a > 0) & (a < 4)]  # 多条件（用 & | ~，不用 and or not）
```

---

### np.where

`np.where(condition, x, y)` — 满足 condition 则取 x，否则取 y：

```python
np.where(a > 0, a, 0)   # ReLU：负数置 0
```

只传 `condition` 时返回满足条件的**索引**：

```python
np.where(a > 0)          # (array([0, 2, 4]),)
```


## 示例 Worked Example

In [ ]:
import numpy as np

# ── 1. 基本切片（view vs copy）────────────────────────────────────────────
a = np.arange(10)
print("原始 a:", a)

v = a[2:8:2]            # view: [2, 4, 6]
print("切片 v (view):", v)
v[0] = 999
print("修改 v[0]=999 后，a:", a)   # a 也变了！

a = np.arange(10)       # 重置
c = a[2:8:2].copy()     # copy
c[0] = 999
print("copy 后修改 c[0]=999，a 不变:", a)


In [ ]:
import numpy as np

# ── 2. 花式索引（Fancy Indexing）──────────────────────────────────────────
rng = np.random.default_rng(0)
A = rng.integers(0, 50, size=(5, 4))
print("矩阵 A:\n", A)

# 选取第 0、2、4 行
rows = A[[0, 2, 4], :]
print("\n选取行 [0,2,4]:\n", rows)

# 同时指定行列
vals = A[[0, 1, 2], [3, 2, 1]]   # A[0,3], A[1,2], A[2,1]
print("\n对角元素选取:", vals)


In [ ]:
import numpy as np

# ── 3. 布尔索引 + np.where ──────────────────────────────────────────────
x = np.array([3, -1, 4, -1, 5, -9, 2, -6])

# 布尔 mask
mask = x > 0
print("正数位置的 mask:", mask)
print("正数元素:", x[mask])

# 多条件
print("0 < x < 5:", x[(x > 0) & (x < 5)])

# np.where（三元操作）：ReLU
relu = np.where(x > 0, x, 0)
print("ReLU(x):", relu)

# 替换：将负数换成 NaN
x_nan = np.where(x > 0, x.astype(float), np.nan)
print("负数→NaN:", x_nan)

# np.where 只传条件 → 返回索引
idx = np.where(x > 0)
print("正数索引:", idx[0])


## 动手 Exercise

给定矩阵 `M`（形状 (6, 4)）：

1. 用布尔索引将所有大于均值的元素提取出来。
2. 用 `np.where` 将大于均值的元素保留，其余置 0。
3. 用花式索引选取第 1、3、5 行，并打印其形状。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
M = rng.integers(0, 100, size=(6, 4))
print("M:\n", M)
print("均值:", M.mean())

# 1. 布尔索引提取大于均值的元素
above_mean = M[M > M.mean()]
print("\n大于均值的元素:", above_mean)

# 2. np.where 保留大于均值的，其余置 0
M_masked = np.where(M > M.mean(), M, 0)
print("\n掩码后矩阵:\n", M_masked)

# 3. 花式索引选取 1、3、5 行
rows_135 = M[[1, 3, 5], :]
print("\n第 1、3、5 行，shape:", rows_135.shape)
print(rows_135)


## 延伸阅读 Further Reading

1. **NumPy 官方文档 — Indexing**: <https://numpy.org/doc/stable/user/basics.indexing.html>
2. **Advanced Indexing**: <https://numpy.org/doc/stable/reference/arrays.indexing.html>
3. **np.where 文档**: <https://numpy.org/doc/stable/reference/generated/numpy.where.html>
4. **视图 vs 拷贝详解**: <https://numpy.org/doc/stable/user/basics.copies.html>


## 关联练习 Related Assignments

本课是 W2 NumPy 基础，对应所有题包：

```bash
python check.py 01-numpy
python check.py 02-broadcasting
```

> 能力标签：**SH3** · threshold ≥ 0.7
